#### Librerías

In [1]:
import requests
import time
import pandas as pd
import json

/Users/aliciagonzalezgozalo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


#### Lista de 30 artistas populares (representando los mercados de Argentina, España y EEUU)

- Uso de una lista de diccionarios ya que es más versátil y permite añadir más campos si fuese necesario
- Los IDs fueron obtenidos a través de la API de Deezer pero la función de búsqueda "search" demostró ser inconsiste, arrojando resultados incorrectos para ciertos artistas. Por ello, hicimos comprobaciones de todos los IDs y aplicamos las correcciones necesarias usando como fuente Deezer.com.

In [ ]:
artists = [
    {"name": "Taylor Swift", "id": 12246},
    {"name": "Drake", "id": 246791},
    {"name": "Morgan Wallen", "id": 7188840},
    {"name": "Kendrick Lamar", "id": 525046},
    {"name": "The Weeknd", "id": 4050205},
    {"name": "SZA", "id": 5531258},
    {"name": "Zach Bryan", "id": 71855892},
    {"name": "Tyler, The Creator", "id": 1194083},
    {"name": "Billie Eilish", "id": 9635624},
    {"name": "Ariana Grande", "id": 1562681},

    {"name": "Bad Bunny", "id": 10583405},
    {"name": "Karol G", "id": 5297021},
    {"name": "Myke Towers", "id": 12029862},
    {"name": "Quevedo", "id": 6705223},
    {"name": "Aitana", "id": 11928391},
    {"name": "Rosalía", "id": 554792},
    {"name": "Morad", "id": 111130212},
    {"name": "Melendi", "id": 2697},
    {"name": "Enrique Iglesias", "id": 2792},
    {"name": "Juanes", "id": 70},

    {"name": "Duki", "id": 4902904},
    {"name": "María Becerra", "id": 14343187},
    {"name": "Tini Stoessel", "id": 2252591},
    {"name": "Nicki Nicole", "id": 13299379},
    {"name": "Cazzu", "id": 12268072},
    {"name": "Lali", "id": 5181388},
    {"name": "Paulo Londra", "id": 12637039},
    {"name": "Shakira", "id": 160},
    {"name": "Bizarrap", "id": 12487862},
    {"name": "Emilia", "id": 61094422}
]

In [ ]:
songs = []

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]

    tracks_request = requests.get(f"https://api.deezer.com/artist/{artist_id}/top?limit=50")

    if tracks_request.status_code != 200:
        print(f"Could not fetch tracks for artist: {artist_name}")
        continue

    tracks_data = tracks_request.json()

    for track in tracks_data["data"]:
        songs.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "track_title": track["title"],
            "album_title": track["album"]["title"],
            "track_type": track["type"]
        })
        
    time.sleep(1)

In [ ]:
song_count = {}

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]

    url = f"https://api.deezer.com/artist/{artist_id}/top?limit=50"
    response = requests.get(url)

    if response.status_code != 200:
        song_count[artist_name] = "Error"
        continue

    data = response.json()
    num_songs = len(data.get("data", []))

    song_count[artist_name] = num_songs

    time.sleep(1)

song_count

{'Taylor Swift': 50,
 'Drake': 50,
 'Morgan Wallen': 37,
 'Kendrick Lamar': 50,
 'The Weeknd': 50,
 'SZA': 50,
 'Zach Bryan': 21,
 'Tyler, The Creator': 50,
 'Billie Eilish': 50,
 'Ariana Grande': 50,
 'Bad Bunny': 50,
 'Karol G': 50,
 'Myke Towers': 50,
 'Quevedo': 50,
 'Aitana': 16,
 'Rosalía': 50,
 'Morad': 19,
 'Melendi': 24,
 'Enrique Iglesias': 50,
 'Juanes': 27,
 'Duki': 50,
 'María Becerra': 45,
 'Tini Stoessel': 37,
 'Nicki Nicole': 26,
 'Cazzu': 21,
 'Lali': 10,
 'Paulo Londra': 48,
 'Shakira': 50,
 'Bizarrap': 50,
 'Emilia': 22}

No todos los artistas tienen 50 canciones en el top de Deezer

In [16]:
songs = []
album_cache = {}

total_artists = len(artists)

for idx, artist in enumerate(artists, start=1):
    artist_id = artist["id"]
    artist_name = artist["name"]

    print(f"\n[{idx}/{total_artists}] Processing artist: {artist_name}")

    # -------------------------------------------------------------------
    # Obtención de información del artista desde la API de Deezer
    # Se utiliza principalmente para obtener metadatos básicos del artista
    # -------------------------------------------------------------------
    artist_info = requests.get(f"https://api.deezer.com/artist/{artist_id}").json()

    # -------------------------------------------------------------------
    # El atributo género se ha dejado como campo pendiente de enriquecimiento,
    # ya que la API de Deezer no proporciona información consistente.
    # Este será completado en la siguiente fase utilizando Last.fm.
    # -------------------------------------------------------------------
    genre = "To be enriched (Last.fm)"
    genre_id = None

    # -------------------------------------------------------------------
    # Obtención de las canciones más populares del artista (hasta 50 tracks)
    # -------------------------------------------------------------------
    response = requests.get(f"https://api.deezer.com/artist/{artist_id}/top?limit=50")

    if response.status_code != 200:
        print(f"Error with {artist_name}")
        continue

    tracks_data = response.json()

    print(f"Tracks retrieved: {len(tracks_data.get('data', []))}")

    for track in tracks_data.get("data", []):

        # -------------------------------------------------------------------
        # Clasificación del tipo de canción:
        # - collab: más de un artista participante
        # - solo: un único artista
        # -------------------------------------------------------------------
        is_collab = len(track.get("contributors", [])) > 1
        track_type = "collab" if is_collab else "solo"

        # -------------------------------------------------------------------
        # Obtención del año de lanzamiento del álbum
        # Se utiliza cache para evitar múltiples llamadas a la API
        # -------------------------------------------------------------------
        album_id = track["album"]["id"]

        if album_id not in album_cache:
            album_info = requests.get(f"https://api.deezer.com/album/{album_id}").json()
            release_date = album_info.get("release_date")
            album_cache[album_id] = release_date[:4] if release_date else None
            time.sleep(0.2)

        release_year = album_cache[album_id]

        # -------------------------------------------------------------------
        # Construcción de la fila final del dataset
        # -------------------------------------------------------------------
        songs.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "track_title": track["title"],
            "album_title": track["album"]["title"],
            "track_type": track_type,
            "release_year": release_year,
            "genre": genre,
            "genre_id": genre_id
        })

    print(f"Finished {artist_name}")

    time.sleep(1)

# -------------------------------------------------------------------
# Creación del DataFrame final para análisis y exportación
# -------------------------------------------------------------------
df = pd.DataFrame(songs)

print("DONE! DataFrame created")

# Visualización completa de columnas
pd.set_option('display.max_columns', None)
df.head()


[1/30] Processing artist: Taylor Swift
Tracks retrieved: 50
Finished Taylor Swift

[2/30] Processing artist: Drake
Tracks retrieved: 50
Finished Drake

[3/30] Processing artist: Morgan Wallen
Tracks retrieved: 37
Finished Morgan Wallen

[4/30] Processing artist: Kendrick Lamar
Tracks retrieved: 50
Finished Kendrick Lamar

[5/30] Processing artist: The Weeknd
Tracks retrieved: 50
Finished The Weeknd

[6/30] Processing artist: SZA
Tracks retrieved: 50
Finished SZA

[7/30] Processing artist: Zach Bryan
Tracks retrieved: 21
Finished Zach Bryan

[8/30] Processing artist: Tyler, The Creator
Tracks retrieved: 50
Finished Tyler, The Creator

[9/30] Processing artist: Billie Eilish
Tracks retrieved: 50
Finished Billie Eilish

[10/30] Processing artist: Ariana Grande
Tracks retrieved: 50
Finished Ariana Grande

[11/30] Processing artist: Bad Bunny
Tracks retrieved: 50
Finished Bad Bunny

[12/30] Processing artist: Karol G
Tracks retrieved: 50
Finished Karol G

[13/30] Processing artist: Myke To

,artist_id,artist_name,track_title,album_title,track_type,release_year,genre,genre_id
0,12246,Taylor Swift,The Fate of Ophelia,The Life of a Showgirl,solo,2025,To be enriched (Last.fm),None
1,12246,Taylor Swift,Opalite,The Life of a Showgirl,solo,2025,To be enriched (Last.fm),None
2,12246,Taylor Swift,Elizabeth Taylor,Elizabeth Taylor,solo,2026,To be enriched (Last.fm),None
3,12246,Taylor Swift,Shake It Off,1989,solo,2014,To be enriched (Last.fm),None
4,12246,Taylor Swift,Eldest Daughter,The Life of a Showgirl,solo,2025,To be enriched (Last.fm),None


In [17]:
df.to_csv("songs_dataset.csv", index=False, encoding="utf-8")